# Visualizzazione Avanzata Risultati e Anatomia Lesionale 3D

Questo notebook permette di caricare un output salvato della pipeline (da `results/`) e visualizzare lo spazio di embedding in 2D e 3D.

Inoltre, consente di selezionare un soggetto e visualizzare la sua lesione cerebrale in un visualizzatore 3D interattivo direttamente nel notebook.

In [ ]:
import sys
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from nilearn import plotting
from IPython.display import display

# Aggiunge la directory root del progetto al path per importare src
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.utils.artifacts import load_matrix
from src.features.clinical import join_lesion_side

print("Librerie importate correttamente.")

## 1. Caricamento del Risultato della Pipeline

Scegliere la cartella dei risultati da ispezionare (dim_reduction o dim_reduction_clustering). Il notebook caricherà l'embedding e i metadati associati.

In [ ]:
# Percorso dei risultati (modificare a piacimento)
results_dir = Path("../results/lesion/dim_reduction/umap/03-08_s1.1_jaccard")

if not results_dir.exists():
    # Fallback su un percorso generico per prevenire errori al primo avvio
    parent_results = Path("../results/lesion/dim_reduction")
    # Troviamo la prima cartella di risultati disponibile
    subdirs = sorted([d for d in parent_results.glob("**/*") if d.is_dir() and (d / "manifest.json").exists()])
    if subdirs:
        results_dir = subdirs[-1]
        print(f"Cartella di fallback trovata: {results_dir}")
    else:
        # Fallback su un altro path potenziale
        parent_results = Path("../results/lesion/dim_reduction_clustering")
        subdirs = sorted([d for d in parent_results.glob("**/*") if d.is_dir() and (d / "manifest.json").exists()])
        if subdirs:
            results_dir = subdirs[-1]
            print(f"Cartella di fallback trovata: {results_dir}")

try:
    embedding, metadata, extra_arrays = load_matrix(results_dir)
    print(f"Risultati caricati con successo da: {results_dir}")
    print(f"Shape dell'embedding: {embedding.shape}")
    print(f"Colonne metadati: {list(metadata.columns)}")
    
    # Assicuriamoci che lesion_side sia presente
    if "lesion_side" not in metadata.columns:
        metadata["lesion_side"] = join_lesion_side(metadata)
        
except Exception as e:
    print(f"Errore: impossibile caricare i risultati da {results_dir}. Assicurarsi che la pipeline abbia prodotto dei risultati. Dettaglio: {e}")

## 2. Visualizzazione 2D degli Embedding

Creiamo scatter plot dell'embedding 2D colorati per le principali variabili (Dataset, Lato Lesione, Volume, e Cluster se presenti).

In [ ]:
if 'embedding' in locals() and embedding.shape[1] >= 2:
    fig, axs = plt.subplots(2, 2, figsize=(12, 9))
    
    # 1. Dataset
    dataset_col = "dataset" if "dataset" in metadata.columns else "dataset_x"
    if dataset_col in metadata.columns:
        import seaborn as sns
        sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], hue=metadata[dataset_col], palette="Set2", s=30, alpha=0.7, ax=axs[0, 0])
        axs[0, 0].set_title("Colorato per Dataset (Batch Effect)")
    
    # 2. Lato Lesione
    if "lesion_side" in metadata.columns:
        import seaborn as sns
        sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], hue=metadata["lesion_side"], palette="Set1", s=30, alpha=0.7, ax=axs[0, 1])
        axs[0, 1].set_title("Colorato per Lato Lesione")
        
    # 3. Volume lesione
    volume_col = "lesion_volume_voxels" if "lesion_volume_voxels" in metadata.columns else "volume_voxel"
    if volume_col in metadata.columns:
        sc = axs[1, 0].scatter(embedding[:, 0], embedding[:, 1], c=metadata[volume_col], cmap="magma", s=25, alpha=0.7)
        axs[1, 0].set_title("Volume Lesionale (Voxel count)")
        fig.colorbar(sc, ax=axs[1, 0], label="Volume (Voxel)")
    else:
        axs[1, 0].text(0.5, 0.5, "Volume non disponibile", ha='center', va='center')
        axs[1, 0].set_title("Volume Lesionale (Voxel count)")
        
    # 4. Cluster Label (se presente)
    if "cluster_label" in metadata.columns:
        import seaborn as sns
        sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], hue=metadata["cluster_label"].astype(str), palette="tab10", s=35, alpha=0.8, ax=axs[1, 1])
        axs[1, 1].set_title("Colorato per Cluster Label")
    else:
        axs[1, 1].text(0.5, 0.5, "Nessun cluster caricato (run di dim reduction pura)", ha='center', va='center')
        axs[1, 1].set_title("Colorato per Cluster Label")
        
    plt.tight_layout()
    plt.show()
else:
    print("Dati non caricati o embedding con meno di 2 dimensioni.")

## 3. Visualizzazione Interattiva in 3D (Plotly)

Se l'embedding contiene 3 o più dimensioni, possiamo esplorare i punti in uno spazio tridimensionale interattivo. Passando il mouse sui punti verranno mostrate le informazioni dettagliate dei soggetti.

In [ ]:
if 'embedding' in locals():
    # Selezioniamo il parametro per colorare l'interactive plot
    color_var = "cluster_label" if "cluster_label" in metadata.columns else "dataset"
    if color_var not in metadata.columns:
        color_var = "dataset_x" if "dataset_x" in metadata.columns else None

    plot_df = metadata.copy()
    plot_df["dim 1"] = embedding[:, 0]
    plot_df["dim 2"] = embedding[:, 1]

    # Se l'embedding ha 3 componenti, mostriamo px.scatter_3d, altrimenti un px.scatter 2D interattivo
    if embedding.shape[1] >= 3:
        plot_df["dim 3"] = embedding[:, 2]
        fig = px.scatter_3d(
            plot_df, 
            x="dim 1", y="dim 2", z="dim 3", 
            color=color_var, 
            hover_data=list(metadata.columns),
            title="Spazio Latente 3D Interattivo"
        )
    else:
        fig = px.scatter(
            plot_df, 
            x="dim 1", y="dim 2", 
            color=color_var, 
            hover_data=list(metadata.columns),
            title="Spazio Latente 2D Interattivo"
        )

    fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
    fig.show()
else:
    print("Dati non caricati.")

## 4. Visualizzatore 3D Interattivo di Anatomia Lesionale (Nilearn)

Selezionando il codice di un soggetto (es. `sub-STUNIPD0001`), cerchiamo il suo file NIfTI originale e lo visualizziamo in 3D overlayato su un cervello standard MNI152.

In [ ]:
if 'metadata' in locals():
    # Inserisci il subject_id da ispezionare
    selected_subject = "sub-STUNIPD0001" 

    # Percorso principale dei dati clinici BIDS
    data_root = Path("../data/clinical_connectome/derivatives")

    # Troviamo il dataset del soggetto cercato
    if selected_subject in metadata["subject_id"].values:
        subject_row = metadata[metadata["subject_id"] == selected_subject].iloc[0]
        dataset_name = subject_row.get("dataset", subject_row.get("dataset_x", None))
        
        if dataset_name:
            # Costruiamo il percorso del manual mask
            lesion_dir = data_root / dataset_name / "manual_masks" / selected_subject / "anat"
            lesion_files = list(lesion_dir.glob("*_label-lesion_mask.nii.gz"))
            
            if lesion_files:
                lesion_path = lesion_files[0]
                print(f"Trovata lesione per {selected_subject} in: {lesion_path}")
                print("Caricamento del visualizzatore 3D interattivo (puoi ruotare, zoomare e selezionare i piani di taglio)...")
                
                # Creiamo il visualizzatore 3D di Nilearn
                view = plotting.view_img(
                    str(lesion_path), 
                    threshold=0.5, 
                    bg_img="MNI152", 
                    title=f"Lesione: {selected_subject} ({dataset_name})"
                )
                # Mostra la view interattiva nel cell output
                display(view)
            else:
                print(f"Errore: Nessun file NIfTI lesion_mask trovato in {lesion_dir}")
        else:
            print(f"Errore: Dataset non trovato nei metadati per {selected_subject}")
    else:
        available_subjects = list(metadata["subject_id"].head(10))
        print(f"Soggetto {selected_subject} non presente in questa matrice.")
        print(f"Esempi di soggetti disponibili nella matrice corrente: {available_subjects}")
else:
    print("Dati non caricati.")